<a href="https://colab.research.google.com/github/viaamanda/Analisis_Deret_Waktu/blob/main/24083010044_Via_Amanda_ADW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import math

from statsmodels.tsa.stattools import adfuller, kpss

In [ ]:
URL_CSV = "https://raw.githubusercontent.com/viaamanda/Analisis_Deret_Waktu/main/harga_komoditas.csv"

# Membaca dataset
df_raw = pd.read_csv(URL_CSV)

display(df_raw.head())

In [ ]:
print("Ukuran dataset:", df_raw.shape)

print("\nNama kolom:")
print(df_raw.columns.tolist())

print("\nJumlah komoditas:",
      df_raw['komoditas'].nunique())


In [ ]:

# ==========================================
# PERSIAPAN DATA AWAL
# ==========================================

data_awal = df_raw.copy()

# Menyeragamkan nama kolom
data_awal.columns = (
    data_awal.columns.str.strip().str.lower()
)

# Mengubah tipe data
data_awal['tanggal'] = pd.to_datetime(
    data_awal['tanggal'],
    errors='coerce'
)

data_awal['harga'] = pd.to_numeric(
    data_awal['harga'],
    errors='coerce'
)

# Membersihkan nama komoditas dan satuan
data_awal['komoditas'] = (
    data_awal['komoditas'].astype('string').str.strip()
)

data_awal['satuan'] = (
    data_awal['satuan'].astype('string').str.strip()
)

# Menghapus data yang tidak valid
data_awal = data_awal.dropna(
    subset=['tanggal', 'komoditas', 'satuan', 'harga']
)

data_awal = data_awal[
    data_awal['harga'] > 0
].copy()

# Mengurutkan data
data_awal = data_awal.sort_values('tanggal')

print("Jumlah observasi:", len(data_awal))
print("Jumlah komoditas:", data_awal['komoditas'].nunique())
print("Periode awal:", data_awal['tanggal'].min())
print("Periode akhir:", data_awal['tanggal'].max())


In [ ]:

# ==========================================
# VISUALISASI STASIONERITAS SELURUH KOMODITAS
# ==========================================

# Mengelompokkan berdasarkan komoditas dan satuan
# agar harga dengan satuan berbeda tidak tercampur

kelompok_komoditas = list(
    data_awal.groupby(['komoditas', 'satuan'])
)

# Pengaturan rolling window
WINDOW = '30D'
MIN_PERIODS = 10

# Jumlah komoditas dalam setiap halaman
PER_HALAMAN = 4

# Menghitung jumlah halaman
jumlah_hal = math.ceil(
    len(kelompok_komoditas) / PER_HALAMAN
)

for halaman in range(jumlah_hal):

    mulai = halaman * PER_HALAMAN
    selesai = mulai + PER_HALAMAN

    kelompok_halaman = kelompok_komoditas[mulai:selesai]

    fig, axes = plt.subplots(
        len(kelompok_halaman),
        2,
        figsize=(17, 4 * len(kelompok_halaman)),
        squeeze=False
    )

    for i, ((nama, satuan), grup) in enumerate(
        kelompok_halaman
    ):

        # Mengurutkan berdasarkan tanggal
        grup = grup.sort_values('tanggal')

        # Jika ada tanggal duplikat,
        # gunakan rata-rata harga per tanggal
        series = (
            grup.groupby('tanggal')['harga']
            .mean()
            .sort_index()
        )

        # Rolling mean dan standard deviation
        rolling_mean = series.rolling(
            WINDOW,
            min_periods=MIN_PERIODS
        ).mean()

        rolling_std = series.rolling(
            WINDOW,
            min_periods=MIN_PERIODS
        ).std()

        # ==================================
        # GRAFIK 1: HARGA DAN ROLLING MEAN
        # ==================================

        ax1 = axes[i, 0]

        ax1.plot(
            series.index,
            series.values,
            label='Harga Aktual'
        )

        ax1.plot(
            rolling_mean.index,
            rolling_mean.values,
            label='Rolling Mean 30 Hari',
            linewidth=2
        )

        ax1.set_title(
            f'{nama} - Harga dan Rolling Mean'
        )

        ax1.set_ylabel(f'Harga ({satuan})')

        ax1.legend(fontsize=8)
        ax1.grid(alpha=0.3)

        # ==================================
        # GRAFIK 2: ROLLING STANDARD DEVIATION
        # ==================================

        ax2 = axes[i, 1]

        ax2.plot(
            rolling_std.index,
            rolling_std.values,
            label='Rolling Std 30 Hari'
        )

        ax2.set_title(
            f'{nama} - Rolling Standard Deviation'
        )

        ax2.set_ylabel(f'Std. Deviasi ({satuan})')

        ax2.legend(fontsize=8)
        ax2.grid(alpha=0.3)

    # Pengaturan tampilan
    fig.suptitle(
        f'Visualisasi Data Awal Komoditas '
        f'(Halaman {halaman + 1}/{jumlah_hal})',
        fontsize=16,
        y=1.01
    )

    plt.tight_layout()
    plt.show()

print("Seluruh visualisasi komoditas selesai.")


In [ ]:
# ==========================================
# FUNGSI UJI STASIONERITAS
# ==========================================

def uji_stasioneritas(series):

    # Uji ADF
    hasil_adf = adfuller(
        series,
        regression='c',
        autolag='AIC'
    )

    adf_statistik = hasil_adf[0]
    adf_pvalue = hasil_adf[1]

    # Uji KPSS
    hasil_kpss = kpss(
        series,
        regression='c',
        nlags='auto'
    )

    kpss_statistik = hasil_kpss[0]
    kpss_pvalue = hasil_kpss[1]

    # Interpretasi ADF
    adf_status = (
        "Stasioner"
        if adf_pvalue < 0.05
        else "Tidak Stasioner"
    )

    # Interpretasi KPSS
    kpss_status = (
        "Stasioner"
        if kpss_pvalue > 0.05
        else "Tidak Stasioner"
    )

    # Kesimpulan gabungan
    if adf_pvalue < 0.05 and kpss_pvalue > 0.05:
        kesimpulan = "Stasioner"

    elif adf_pvalue > 0.05 and kpss_pvalue < 0.05:
        kesimpulan = "Tidak Stasioner"

    elif adf_pvalue < 0.05 and kpss_pvalue < 0.05:
        kesimpulan = "Hasil Berbeda"

    else:
        kesimpulan = "Belum Konklusif"

    return {
        'ADF_Statistik': adf_statistik,
        'ADF_pvalue': adf_pvalue,
        'ADF_Status': adf_status,
        'KPSS_Statistik': kpss_statistik,
        'KPSS_pvalue': kpss_pvalue,
        'KPSS_Status': kpss_status,
        'Kesimpulan': kesimpulan
    }

In [ ]:
# UJI STASIONERITAS SELURUH KOMODITAS

hasil_pengujian = []

for komoditas, grup in data_awal.groupby('komoditas'):

    # Mengurutkan data berdasarkan tanggal
    grup = grup.sort_values('tanggal')

    # Memeriksa konsistensi satuan
    if grup['satuan'].nunique() != 1:
        hasil_pengujian.append({
            'Komoditas': komoditas,
            'Jumlah_Observasi': len(grup),
            'Jumlah_Missing_Date': np.nan,
            'ADF_pvalue': np.nan,
            'KPSS_pvalue': np.nan,
            'Kesimpulan': 'Satuan Tidak Konsisten'
        })
        continue

    # Menggabungkan duplikasi tanggal dengan rata-rata
    series = grup.groupby('tanggal')['harga'].mean()

    # Membuat indeks harian lengkap
    tanggal_lengkap = pd.date_range(
        start=series.index.min(),
        end=series.index.max(),
        freq='D'
    )

    series = series.reindex(tanggal_lengkap)

    # Menghitung tanggal yang tidak tercatat
    jumlah_missing = series.isna().sum()

    # Interpolasi maksimal 3 hari berturut-turut
    series = series.interpolate(
        method='time',
        limit=3,
        limit_area='inside'
    )

    # Jika masih ada missing, jangan dipaksakan
    if series.isna().any():

        hasil_pengujian.append({
            'Komoditas': komoditas,
            'Jumlah_Observasi': len(series),
            'Jumlah_Missing_Date': jumlah_missing,
            'ADF_pvalue': np.nan,
            'KPSS_pvalue': np.nan,
            'Kesimpulan': 'Gap Data Terlalu Panjang'
        })

        continue

    # Minimal observasi
    if len(series) < 30:

        hasil_pengujian.append({
            'Komoditas': komoditas,
            'Jumlah_Observasi': len(series),
            'Jumlah_Missing_Date': jumlah_missing,
            'ADF_pvalue': np.nan,
            'KPSS_pvalue': np.nan,
            'Kesimpulan': 'Observasi Tidak Cukup'
        })

        continue

    try:

        hasil = uji_stasioneritas(series.values)

        hasil_pengujian.append({
            'Komoditas': komoditas,
            'Jumlah_Observasi': len(series),
            'Jumlah_Missing_Date': jumlah_missing,
            **hasil
        })

    except Exception as e:

        hasil_pengujian.append({
            'Komoditas': komoditas,
            'Jumlah_Observasi': len(series),
            'Jumlah_Missing_Date': jumlah_missing,
            'ADF_pvalue': np.nan,
            'KPSS_pvalue': np.nan,
            'Kesimpulan': f'Gagal diuji: {str(e)[:60]}'
        })


# Mengubah hasil menjadi DataFrame
hasil_stasioneritas = pd.DataFrame(hasil_pengujian)

# Mengurutkan berdasarkan nama komoditas
hasil_stasioneritas = hasil_stasioneritas.sort_values(
    'Komoditas'
).reset_index(drop=True)

display(hasil_stasioneritas)

In [ ]:
# MENAMPILKAN SELURUH KOMODITAS TIDAK STASIONER

# Mengatur agar seluruh baris dan kolom terlihat
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Mengambil komoditas yang tidak stasioner
komoditas_tidak_stasioner = hasil_stasioneritas[
    hasil_stasioneritas['Kesimpulan'] == 'Tidak Stasioner'
].copy()

# Mengurutkan berdasarkan nama komoditas
komoditas_tidak_stasioner = (
    komoditas_tidak_stasioner
    .sort_values('Komoditas')
    .reset_index(drop=True)
)

# Menampilkan jumlah komoditas
print("DAFTAR SELURUH KOMODITAS TIDAK STASIONER")

print(
    f"Total komoditas tidak stasioner: "
    f"{len(komoditas_tidak_stasioner)}"
)

# Menampilkan seluruh hasil tanpa pemotongan
print(
    komoditas_tidak_stasioner.to_string(index=False)
)

# PENGAMBILAN DATA KOMODITAS KACANG HIJAU

In [ ]:
# Membuat salinan dataset
data_kacang_hijau = df_raw.copy()

# Menyeragamkan nama kolom
data_kacang_hijau.columns = (
    data_kacang_hijau.columns.str.strip().str.lower()
)

# Mengubah format tanggal
data_kacang_hijau['tanggal'] = pd.to_datetime(
    data_kacang_hijau['tanggal'],
    errors='coerce'
)

# Mengubah harga menjadi numerik
data_kacang_hijau['harga'] = pd.to_numeric(
    data_kacang_hijau['harga'],
    errors='coerce'
)

# Membersihkan nama komoditas
data_kacang_hijau['komoditas'] = (
    data_kacang_hijau['komoditas']
    .astype('string')
    .str.strip()
)

# Mengambil hanya komoditas Kacang Hijau
data_kacang_hijau = data_kacang_hijau[
    data_kacang_hijau['komoditas'].str.casefold()
    == 'kacang hijau'
].copy()

# Membersihkan data tidak valid
data_kacang_hijau = data_kacang_hijau.dropna(
    subset=['tanggal', 'harga', 'satuan']
)

data_kacang_hijau = data_kacang_hijau[
    data_kacang_hijau['harga'] > 0
].copy()

# Mengurutkan berdasarkan tanggal
data_kacang_hijau = data_kacang_hijau.sort_values(
    'tanggal'
).reset_index(drop=True)

print("Seleksi komoditas berhasil!")
print("Nama komoditas:",
      data_kacang_hijau['komoditas'].unique())

print("Jumlah observasi:",
      len(data_kacang_hijau))

In [ ]:
data_kacang_hijau.head()

In [ ]:
data_kacang_hijau.tail()

In [ ]:
data_kacang_hijau.info()

In [ ]:
data_kacang_hijau.describe()

In [ ]:
# MEMBENTUK DATA TIME SERIES

time_series_kacang_hijau = (
    data_kacang_hijau
    .set_index('tanggal')['harga']
    .sort_index()
)

print("Time series Kacang Hijau berhasil dibuat.")

print("Jumlah observasi:",
      len(time_series_kacang_hijau))

display(time_series_kacang_hijau.head())

In [ ]:

# ==========================================
# VISUALISASI PERGERAKAN HARGA KACANG HIJAU
# ==========================================

# Membuat salinan data
data_visualisasi = data_kacang_hijau.copy()

# Memastikan format tanggal dan harga
data_visualisasi['tanggal'] = pd.to_datetime(
    data_visualisasi['tanggal']
)

data_visualisasi['harga'] = pd.to_numeric(
    data_visualisasi['harga'],
    errors='coerce'
)

# Mengurutkan data berdasarkan tanggal
data_visualisasi = data_visualisasi.sort_values('tanggal')

# Menggabungkan jika terdapat tanggal duplikat
series_harga = (
    data_visualisasi
    .groupby('tanggal')['harga']
    .mean()
    .sort_index()
)

# Membuat kalender harian lengkap
tanggal_lengkap = pd.date_range(
    start=series_harga.index.min(),
    end=series_harga.index.max(),
    freq='D'
)

series_lengkap = series_harga.reindex(tanggal_lengkap)
series_lengkap.index.name = 'tanggal'

# Menghitung rolling mean 30 hari
rolling_mean = series_lengkap.rolling(
    window='30D',
    min_periods=10
).mean()

# Informasi dataset
jumlah_observasi = len(series_harga)
jumlah_missing = series_lengkap.isna().sum()

tanggal_awal = series_harga.index.min()
tanggal_akhir = series_harga.index.max()

satuan = data_visualisasi['satuan'].mode().iloc[0]

# ==========================================
# MEMBUAT GRAFIK
# ==========================================

plt.figure(figsize=(16, 7))

plt.plot(
    series_lengkap.index,
    series_lengkap.values,
    label='Harga Aktual',
    linewidth=1
)

plt.plot(
    rolling_mean.index,
    rolling_mean.values,
    label='Rolling Mean (30 Hari)',
    linewidth=2
)

plt.title(
    'Pergerakan Harga Kacang Hijau (2023–2026)',
    fontsize=16
)

plt.xlabel('Tanggal', fontsize=12)
plt.ylabel(f'Harga ({satuan})', fontsize=12)

plt.legend()
plt.grid(alpha=0.3)
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

print("Jumlah observasi aktual:", jumlah_observasi)
print("Jumlah missing date:", jumlah_missing)
print("Periode:", tanggal_awal.date(), "sampai", tanggal_akhir.date())


In [ ]:

# ==========================================
# HISTOGRAM DISTRIBUSI HARGA
# ==========================================

harga = series_harga.dropna()

rata_rata = harga.mean()
median = harga.median()

plt.figure(figsize=(12, 6))

plt.hist(
    harga,
    bins=30,
    edgecolor='black',
    alpha=0.75
)

plt.axvline(
    rata_rata,
    linestyle='--',
    linewidth=2,
    label=f'Rata-rata: Rp{rata_rata:,.2f}'
)

plt.axvline(
    median,
    linestyle=':',
    linewidth=2,
    label=f'Median: Rp{median:,.2f}'
)

plt.title(
    'Distribusi Harga Kacang Hijau (2023–2026)',
    fontsize=15
)

plt.xlabel(f'Harga ({satuan})')
plt.ylabel('Frekuensi')

plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# BOXPLOT HARGA KACANG HIJAU

plt.figure(figsize=(12, 4))

plt.boxplot(
    harga,
    vert=False,
    showmeans=True
)

plt.title(
    'Boxplot Harga Kacang Hijau (2023–2026)',
    fontsize=15
)

plt.xlabel(f'Harga ({satuan})')
plt.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()
